# Chapter 36 — Large Language Models: Tokens, Decoding, Scaling

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch36/_lib.py`.

In [2]:
import numpy as np, warnings, re; warnings.filterwarnings("ignore")
from collections import Counter, defaultdict

def make_corpus(n_docs, seed):
    """Synthetic customer-review-style text from a hand-built grammar with
    realistic word frequencies (Zipfian), so tokenization, decoding, and
    scaling can be demonstrated on text with no copyright status question."""
    r = np.random.default_rng(seed)
    subjects = ["the product", "this item", "the device", "the service", "delivery",
                "the packaging", "the battery", "the screen", "support", "the app"]
    verbs = ["arrived", "worked", "broke", "improved", "failed", "lasted",
             "shipped", "performed", "exceeded", "disappointed"]
    adverbs = ["quickly", "slowly", "eventually", "immediately", "barely",
               "consistently", "rarely", "surprisingly", "finally", "never"]
    objects = ["expectations", "the first week", "two days", "a month",
               "the warranty period", "every test", "the price point",
               "my needs", "the description", "the competition"]
    connectors = ["and", "but", "so", "because", "although", "while"]

    def weighted(options, alpha=1.3):
        w = np.array([1 / (i + 1) ** alpha for i in range(len(options))])
        return r.choice(options, p=w / w.sum())

    docs = []
    for _ in range(n_docs):
        n_sent = r.integers(1, 4)
        sentences = []
        for _ in range(n_sent):
            s = f"{weighted(subjects)} {weighted(verbs)} {weighted(adverbs)}"
            if r.random() < 0.6:
                s += f" {weighted(connectors)} {weighted(subjects)} {weighted(verbs)} {weighted(objects)}"
            sentences.append(s)
        docs.append(". ".join(sentences) + ".")
    return docs

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# Byte-pair encoding builds a vocabulary bottom-up: start from individual
# characters, and repeatedly merge whichever adjacent pair appears most
# often, treating that pair as one new unit from then on.
def get_pair_counts(word_freqs):
    pairs = Counter()
    for word, freq in word_freqs.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i + 1])] += freq
    return pairs

def merge_pair(pair, word_freqs):
    merged = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, freq in word_freqs.items():
        merged[word.replace(bigram, replacement)] = freq
    return merged

corpus = make_corpus(200, seed=36)
text = " ".join(corpus)
words = text.split()
# start as characters
word_freqs = Counter(" ".join(list(w)) + " </w>" for w in words)

print(f"corpus: {len(words)} word occurrences, {len(word_freqs)} "
      f"distinct words")
print(f"starting vocabulary: "
      f"{len(set(c for w in word_freqs for c in w.split()))} characters")

merges = []
vocab_over_time = []
wf = dict(word_freqs)
for step in range(30):
    pairs = get_pair_counts(wf)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    wf = merge_pair(best, wf)
    merges.append((best, pairs[best]))
    vocab_size = len(set(c for w in wf for c in w.split()))
    vocab_over_time.append(vocab_size)

print(f"\nfirst eight merges, most frequent adjacent pair each round:")
for i, (pair, count) in enumerate(merges[:8], 1):
    print(f"  {i}. {pair[0]!r} + {pair[1]!r}  (seen together {count} times)")

corpus: 3044 word occurrences, 65 distinct words
starting vocabulary: 26 characters

first eight merges, most frequent adjacent pair each round:
  1. 'e' + '</w>'  (seen together 814 times)
  2. 't' + 'h'  (seen together 728 times)
  3. 'd' + '</w>'  (seen together 722 times)
  4. 'e' + 'd</w>'  (seen together 594 times)
  5. 'th' + 'e</w>'  (seen together 589 times)
  6. 'r' + 'o'  (seen together 421 times)
  7. '.' + '</w>'  (seen together 416 times)
  8. 'l' + 'y'  (seen together 413 times)


### Block 2  (`c2.py`)

In [4]:
# A larger vocabulary means fewer tokens per sentence, at the cost of a
# larger table of units the model must represent and predict over.
def apply_merges(text, merges):
    tokens = list(text)
    words_with_boundary = []
    for w in text.split(" "):
        words_with_boundary.append(list(w) + ["</w>"])
    for pair, _ in merges:
        for wi, w in enumerate(words_with_boundary):
            new_w, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == pair[0] and w[i+1] == pair[1]:
                    new_w.append(w[i] + w[i+1]); i += 2
                else:
                    new_w.append(w[i]); i += 1
            words_with_boundary[wi] = new_w
    return [tok for w in words_with_boundary for tok in w]

test_sentence = ("the product arrived quickly and this item "
                 "worked consistently.")

print(f"{'merges applied':>15}{'vocabulary size':>18}"
      f"{'tokens for one sentence':>26}")
for n_merges in (0, 5, 10, 20, 30):
    partial_merges = merges[:n_merges]
    toks = apply_merges(test_sentence, partial_merges)
    vocab = (len(set(c for w in wf for c in w.split()))
             if n_merges == len(merges) else None)
    # 26 letters + merges + boundary marker, roughly
    vocab_size = 26 + n_merges + 1
    print(f"{n_merges:>15}"
          f"{vocab_over_time[n_merges-1] if n_merges else 26:>18}"
          f"{len(toks):>26}")

print(f"\nthe sentence: {test_sentence!r}")
print(f"at 0 merges (characters):  {apply_merges(test_sentence, [])}")
print(f"at 20 merges:              "
      f"{apply_merges(test_sentence, merges[:20])}")

 merges applied   vocabulary size   tokens for one sentence
              0                26                        63
              5                31                        54
             10                35                        48
             20                42                        38
             30                50                        28

the sentence: 'the product arrived quickly and this item worked consistently.'
at 0 merges (characters):  ['t', 'h', 'e', '</w>', 'p', 'r', 'o', 'd', 'u', 'c', 't', '</w>', 'a', 'r', 'r', 'i', 'v', 'e', 'd', '</w>', 'q', 'u', 'i', 'c', 'k', 'l', 'y', '</w>', 'a', 'n', 'd', '</w>', 't', 'h', 'i', 's', '</w>', 'i', 't', 'e', 'm', '</w>', 'w', 'o', 'r', 'k', 'e', 'd', '</w>', 'c', 'o', 'n', 's', 'i', 's', 't', 'e', 'n', 't', 'l', 'y', '.', '</w>']
at 20 merges:              ['the</w>', 'product</w>', 'arrived</w>', 'q', 'u', 'ic', 'k', 'ly', '</w>', 'a', 'n', 'd</w>', 'th', 'i', 's', '</w>', 'i', 't', 'e', 'm', '</w>', 'w', 'o', 'r', 

### Block 3  (`c3.py`)

In [5]:
# A next-token distribution to decode from. A trigram model: predict the
# next word from the two words before it, estimated by counting.
def train_ngram(docs, n=3):
    counts = defaultdict(Counter)
    for doc in docs:
        words = ["<s>"] * (n - 1) + doc.split() + ["</s>"]
        for i in range(len(words) - n + 1):
            context = tuple(words[i:i + n - 1])
            counts[context][words[i + n - 1]] += 1
    return counts

def next_word_probs(counts, context, vocab, smoothing=0.1):
    c = counts.get(context, Counter())
    total = sum(c.values()) + smoothing * len(vocab)
    return {w: (c.get(w, 0) + smoothing) / total for w in vocab}

train_docs = make_corpus(400, seed=36)
vocab = sorted(set(w for d in train_docs for w in d.split()) | {"</s>"})
counts = train_ngram(train_docs, n=3)

context = ("the", "product")
probs = next_word_probs(counts, context, vocab)
top5 = sorted(probs.items(), key=lambda x: -x[1])[:5]
print(f"trained on {len(train_docs)} synthetic reviews, vocabulary of "
      f"{len(vocab)} words")
print(f"\nafter the context {context}, most likely next words:")
for w, p in top5:
    print(f"  {w:<14}{p:.4f}")
print(f"\ntotal probability mass: {sum(probs.values()):.4f}   "
      f"(must sum to 1)")

trained on 400 synthetic reviews, vocabulary of 66 words

after the context ('the', 'product'), most likely next words:
  arrived       0.4352
  worked        0.2026
  improved      0.0905
  broke         0.0889
  failed        0.0403

total probability mass: 1.0000   (must sum to 1)


### Block 4  (`c4.py`)

In [6]:
# Four ways to turn a probability distribution into an actual next word.
def sample_from(probs_dict, method, seed, temperature=1.0, k=5, p=0.9):
    r = np.random.default_rng(seed)
    words = list(probs_dict.keys())
    p_arr = np.array([probs_dict[w] for w in words])

    if method == "greedy":
        return words[p_arr.argmax()]
    if method == "temperature":
        logp = np.log(p_arr + 1e-12) / temperature
        adj = np.exp(logp - logp.max()); adj /= adj.sum()
        return r.choice(words, p=adj)
    if method == "top_k":
        idx = np.argsort(-p_arr, kind="stable")[:k]
        sub = p_arr[idx]; sub /= sub.sum()
        return r.choice([words[i] for i in idx], p=sub)
    if method == "top_p":
        order = np.argsort(-p_arr, kind="stable")
        cum = np.cumsum(p_arr[order])
        cutoff = np.searchsorted(cum, p) + 1
        idx = order[:cutoff]
        sub = p_arr[idx]; sub /= sub.sum()
        return r.choice([words[i] for i in idx], p=sub)

def generate(counts, vocab, method, seed, max_len=12, **kw):
    r_start = np.random.default_rng(seed)
    words = ["<s>", "<s>"]
    for i in range(max_len):
        context = tuple(words[-2:])
        probs = next_word_probs(counts, context, vocab)
        nxt = sample_from(probs, method, seed=seed * 1000 + i, **kw)
        if nxt == "</s>":
            break
        words.append(nxt)
    return " ".join(words[2:])

print("greedy (always the single most likely word):")
print(" ", generate(counts, vocab, "greedy", seed=36))
print(" ", generate(counts, vocab, "greedy", seed=37))
print(" ", generate(counts, vocab, "greedy", seed=38))

print("\ntemperature = 0.7 (sample, mildly reshaped toward the top):")
for s in (36, 37, 38):
    print(" ", generate(counts, vocab, "temperature", seed=s,
                        temperature=0.7))

print("\ntop-k = 3 (sample among the three most likely words only):")
for s in (36, 37, 38):
    print(" ", generate(counts, vocab, "top_k", seed=s, k=3))

print("\ntop-p = 0.9 (sample among the smallest set covering 90% mass):")
for s in (36, 37, 38):
    print(" ", generate(counts, vocab, "top_p", seed=s, p=0.9))

greedy (always the single most likely word):
  the product arrived quickly and the product arrived quickly and the product
  the product arrived quickly and the product arrived quickly and the product
  the product arrived quickly and the product arrived quickly and the product

temperature = 0.7 (sample, mildly reshaped toward the top):
  the product worked quickly and this item worked quickly.
  the product arrived quickly and the product arrived the first week.
  this item worked every test. the screen arrived expectations. the product worked

top-k = 3 (sample among the three most likely words only):
  the device broke the first week. the service worked expectations.
  the device arrived quickly. the product worked quickly but the service arrived
  delivery broke eventually and the device broke expectations.

top-p = 0.9 (sample among the smallest set covering 90% mass):
  the device failed quickly.
  the service arrived expectations. the product broke expectations. finally point. 

### Block 5  (`c5.py`)

In [7]:
# Quantify what "repetitive" actually means: the fraction of generated
# bigrams that are exact repeats of an earlier bigram in the same
# generation, averaged over many independent generations.
def repetition_rate(counts, vocab, method, n_runs=40, **kw):
    rates = []
    for i in range(n_runs):
        text = generate(counts, vocab, method, seed=1000 + i,
                        max_len=20, **kw)
        toks = text.split()
        if len(toks) < 3:
            continue
        bigrams = list(zip(toks, toks[1:]))
        seen, repeats = set(), 0
        for b in bigrams:
            if b in seen:
                repeats += 1
            seen.add(b)
        rates.append(repeats / max(len(bigrams), 1))
    return np.mean(rates)

print(f"{'method':<16}{'mean bigram repetition rate':>30}")
for method, kw in [("greedy", {}),
                   ("temperature (0.7)", {"temperature": 0.7}),
                   ("top-k (3)", {"k": 3}),
                   ("top-p (0.9)", {"p": 0.9})]:
    key = ("temperature" if "temperature" in method else
           "top_k" if "top-k" in method else
           "top_p" if "top-p" in method else "greedy")
    rate = repetition_rate(counts, vocab, key, **kw)
    print(f"{method:<16}{rate:>30.4f}")

method             mean bigram repetition rate
greedy                                  0.7368
temperature (0.7)                        0.1243
top-k (3)                               0.0941
top-p (0.9)                             0.0223


### Block 6  (`c6.py`)

In [8]:
# Scaling laws, at a scale small enough to run in seconds rather than
# GPU-months: does more training data reduce a language model's error,
# and does that benefit run out? Perplexity is the standard metric,
# the exponential of the average negative log-likelihood per word.
def perplexity(counts, vocab, docs, n=3):
    total_logp, total_words = 0.0, 0
    for doc in docs:
        words = ["<s>"] * (n - 1) + doc.split() + ["</s>"]
        for i in range(n - 1, len(words)):
            context = tuple(words[i - n + 1:i])
            probs = next_word_probs(counts, context, vocab)
            total_logp += np.log(probs.get(words[i], 1e-12))
            total_words += 1
    return np.exp(-total_logp / total_words)

# fixed evaluation set, never trained on
held_out = make_corpus(300, seed=999)
all_docs = train_docs + held_out
vocab_full = sorted(set(w for d in all_docs for w in d.split()) | {"</s>"})

print(f"{'training documents':>19}{'perplexity on held-out text':>30}")
for n_docs in (10, 30, 100, 300, 1000, 3000):
    docs_n = make_corpus(n_docs, seed=36)
    counts_n = train_ngram(docs_n, n=3)
    ppl = perplexity(counts_n, vocab_full, held_out)
    print(f"{n_docs:>19}{ppl:>30.2f}")

print(f"\nfor reference, a model assigning equal probability to all ")
print(f"{len(vocab_full)} words would score a perplexity of "
      f"{len(vocab_full)}.")

 training documents   perplexity on held-out text


                 10                         24.82


                 30                         15.44


                100                          9.77


                300                          6.94


               1000                          5.49


               3000                          4.86

for reference, a model assigning equal probability to all 
66 words would score a perplexity of 66.


### Block 7  (`c7.py`)

In [9]:
# Capacity interacts with data. A higher-order model (more context,
# more parameters in the count table) can represent more, but each
# additional context needs its own examples: too little data and a
# larger model overfits to noise in its own counts.
print(f"{'training docs':>14}{'bigram (n=2)':>14}{'trigram (n=3)':>15}"
      f"{'4-gram (n=4)':>14}")
for n_docs in (10, 50, 300, 2000):
    docs_n = make_corpus(n_docs, seed=36)
    row = [n_docs]
    for order in (2, 3, 4):
        counts_n = train_ngram(docs_n, n=order)
        ppl = perplexity(counts_n, vocab_full, held_out, n=order)
        row.append(ppl)
    print(f"{row[0]:>14}{row[1]:>14.2f}{row[2]:>15.2f}{row[3]:>14.2f}")

print(f"\nmore context helps once there is enough data to fill it in;")
print(f"with too little data, the extra context has nothing reliable")
print(f"to have learned, and can score worse than the simpler model.")

 training docs  bigram (n=2)  trigram (n=3)  4-gram (n=4)
            10         14.57          24.82         36.97


            50          7.69          12.48         22.24
           300          5.37           6.94         11.47


          2000          4.86           5.03          6.56

more context helps once there is enough data to fill it in;
with too little data, the extra context has nothing reliable
to have learned, and can score worse than the simpler model.
